# 04 SHAP and substructure analysis

Work out what Model A actually learned. SHAP ranks which fingerprint bits drive the inhibition calls, then the bit info map turns each of those bits back into real atoms on real molecules, so a bit index becomes a chemical motif a reader can recognise.

SHAP only says the model leaned on a motif. A Fisher exact test per motif, with Benjamini-Hochberg correction across all the motifs tested, says whether that motif is genuinely enriched among inhibitors or just turned up by chance. Both are needed: a motif can dominate the model and still be an artefact of the training set.

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join("..", "src")))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from rdkit import Chem
from IPython.display import SVG, display

import features
import explain
import stats

PROCESSED_DIR = os.path.join("..", "data", "processed")

## Load Model A and the data it was fitted on

In [ ]:
model_a = joblib.load(os.path.join(PROCESSED_DIR, "model_a.joblib"))
X = np.load(os.path.join(PROCESSED_DIR, "X_cyp3a4.npy"))
y = np.load(os.path.join(PROCESSED_DIR, "y_cyp3a4.npy"))
cyp_df = pd.read_csv(os.path.join(PROCESSED_DIR, "cyp3a4_clean.csv"))
split = np.load(os.path.join(PROCESSED_DIR, "split_a.npz"))

smiles = cyp_df["Drug"].tolist()
print("X:", X.shape, " positives:", int(y.sum()))
print("model:", model_a.n_estimators, "trees")

## SHAP

`TreeExplainer` is exact for tree models rather than an approximation, which is
the reason a Random Forest was worth keeping over something less interpretable.

It is run on a **subsample of the held out rows**, for two reasons: TreeExplainer
over 12k rows x 500 trees is slow, and explaining the model on rows it was fitted
on would describe how it memorised the training set rather than how it
generalises. The bit ranking barely moves above a few hundred rows.

In [ ]:
rng = np.random.default_rng(42)
test_idx = split["idx_test"]
sample_idx = rng.choice(test_idx, size=min(400, len(test_idx)), replace=False)
X_sample = X[sample_idx]

print(f"explaining {X_sample.shape[0]} held out compounds")
top_bits, top_values, shap_values = explain.explain_model(model_a, X_sample, top_n=25)

print(f"SHAP matrix: {shap_values.shape}")
print()
print("top 25 bits by mean absolute SHAP value:")
for bit, value in zip(top_bits, top_values):
    print(f"   bit {int(bit):4d}   {value:.5f}")

In [ ]:
import shap

# The summary plot for the report. Each row is a fingerprint bit, each dot a
# compound, coloured by whether the bit is set.
shap.summary_plot(
    shap_values, X_sample,
    feature_names=[f"bit {i}" for i in range(X_sample.shape[1])],
    max_display=20, show=False,
)
plt.title("Model A: fingerprint bits driving CYP3A4 inhibition calls")
plt.tight_layout()
plt.show()

## From bit index to chemistry

A bit index tells a reader nothing on its own. `explain.bit_to_atoms` reruns the
**shared** Morgan generator from `features.py` with a bit info map, so the bit
numbering matches the training data exactly, then expands each bit's
(centre atom, radius) back into the atoms it covers.

`describe_fragment` reads the chemistry flags off the parent molecule's atoms.
It does not write the fragment out as SMILES and re-parse it: a fragment such as
`ccc` is not valid standalone SMILES, so that route returns `None` and reports
nothing while looking like it worked.

In [ ]:
# Find real inhibitors that actually contain each top bit, so the motifs are
# shown on molecules rather than in the abstract.
examples = explain.top_bit_examples(smiles, y, top_bits[:8], X, n_examples=3)

for bit, rows in examples.items():
    if not rows:
        print(f"bit {bit}: no positive example found")
        continue
    row_idx, smi = rows[0]
    mol = Chem.MolFromSmiles(smi)
    atom_map = explain.bit_to_atoms(smi, [bit])
    atoms = atom_map.get(bit, [])
    description = explain.describe_fragment(mol, atoms)
    print(f"bit {bit:4d}  ->  {description['summary']}")
    print(f"           example: {smi[:70]}")
    print()

In [ ]:
# Draw the top few motifs highlighted on an inhibitor that contains them.
for bit in top_bits[:5]:
    rows = examples.get(int(bit), [])
    if not rows:
        continue
    row_idx, smi = rows[0]
    mol = Chem.MolFromSmiles(smi)
    atoms = explain.bit_to_atoms(smi, [int(bit)]).get(int(bit), [])
    if not atoms:
        continue
    description = explain.describe_fragment(mol, atoms)
    print(f"bit {int(bit)}: {description['summary']}")
    display(SVG(explain.draw_fragment(mol, atoms)))

## Is the motif really enriched?

Fisher's exact test per bit, then Benjamini-Hochberg across every bit tested.

The correction matters: one test per fingerprint bit over 1024 bits would hand
us roughly 50 false hits at alpha 0.05 even if no motif mattered at all. The
`n_tests` column records how many bits the correction was applied across,
because the same raw p value adjusts differently over 25 tests than over 1024.

In [ ]:
# The top SHAP bits first: these are the ones the report will discuss.
table = stats.enrichment_table(X, y, [int(b) for b in top_bits], alpha=0.05)

print(f"{int(table['significant'].sum())} of {len(table)} top SHAP bits are "
      f"significantly enriched after BH correction")
print()
print(table[["bit", "odds_ratio", "p_value", "p_adjusted", "significant",
             "n_present", "present_pos", "present_neg"]].to_string(index=False))

In [ ]:
# Now every bit, as the honest denominator. A bit that looks significant among
# 25 hand-picked bits should still look significant when the correction is
# applied across all 1024.
all_bits = [b for b in range(X.shape[1]) if X[:, b].sum() >= 10]
print(f"testing {len(all_bits)} bits that appear in at least 10 compounds")

full_table = stats.enrichment_table(X, y, all_bits, alpha=0.05)
full_table.to_csv(os.path.join(PROCESSED_DIR, "motif_enrichment.csv"), index=False)

print(f"significant after BH across all {len(all_bits)} bits: "
      f"{int(full_table['significant'].sum())}")
print()
print("top 20 enriched motifs overall:")
print(full_table.head(20)[["bit", "odds_ratio", "p_value", "p_adjusted",
                           "n_present"]].to_string(index=False))

In [ ]:
# Which of the SHAP-important bits survive the full correction? A bit the model
# leans on heavily that is NOT enriched is worth reporting: it means the model
# found something that is not a simple one-motif association.
shap_set = set(int(b) for b in top_bits)
overlap = full_table[full_table["bit"].isin(shap_set)]
survived = overlap[overlap["significant"]]

print(f"of the {len(shap_set)} top SHAP bits, {len(survived)} are significant")
print(f"when corrected across all {len(all_bits)} tested bits")
print()
not_sig = overlap[~overlap["significant"]]
if len(not_sig):
    print("SHAP-important but NOT significantly enriched (report these):")
    print(not_sig[["bit", "odds_ratio", "p_adjusted"]].to_string(index=False))

## Sanity check against known CYP3A4 pharmacology

CYP3A4 has a large, hydrophobic active site, so its inhibitors are
characteristically **big, lipophilic and aromatic**. If the motifs the model
leans on point that way, that is weak external support. If they point somewhere
else entirely, the model may be reading a quirk of the Veith assay instead.

This is a direction check on aggregate properties, not a claim about any one
compound.

In [ ]:
from rdkit.Chem import Descriptors, Crippen

# Compare predicted inhibitors against predicted non-inhibitors on the
# properties the literature associates with CYP3A4 inhibition.
probs = model_a.predict_proba(X[test_idx])[:, 1]
test_smiles = [smiles[i] for i in test_idx]

records = []
for smi, prob in zip(test_smiles, probs):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue
    records.append({
        "prob": prob,
        "mol_weight": Descriptors.MolWt(mol),
        "logp": Crippen.MolLogP(mol),
        "aromatic_rings": Descriptors.NumAromaticRings(mol),
        "tpsa": Descriptors.TPSA(mol),
    })

props = pd.DataFrame(records)
predicted_inhibitor = props["prob"] >= 0.70
predicted_not = props["prob"] <= 0.30

print(f"predicted inhibitors: {predicted_inhibitor.sum()}, "
      f"predicted non-inhibitors: {predicted_not.sum()}")
print()
comparison = pd.DataFrame({
    "predicted inhibitor": props[predicted_inhibitor].mean(numeric_only=True),
    "predicted non-inhibitor": props[predicted_not].mean(numeric_only=True),
})
comparison["difference"] = (comparison["predicted inhibitor"]
                            - comparison["predicted non-inhibitor"])
print(comparison.drop(index="prob").to_string())
print()
print("Expected direction if the model matches known CYP3A4 pharmacology:")
print("  mol_weight     higher in inhibitors")
print("  logp           higher in inhibitors (more lipophilic)")
print("  aromatic_rings higher in inhibitors")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, column, label in zip(
    axes,
    ["mol_weight", "logp", "aromatic_rings"],
    ["molecular weight", "cLogP", "aromatic rings"],
):
    ax.hist(props[predicted_not][column], bins=30, alpha=0.6,
            label="predicted non-inhibitor", density=True)
    ax.hist(props[predicted_inhibitor][column], bins=30, alpha=0.6,
            label="predicted inhibitor", density=True)
    ax.set_xlabel(label)
    ax.set_ylabel("density")
    ax.legend(fontsize=8)
fig.suptitle("Do Model A's calls line up with known CYP3A4 inhibitor properties?")
plt.tight_layout()
plt.show()

## What this section can and cannot claim

- SHAP says which bits the model **used**. It is not evidence that the motif
  causes inhibition.
- A significant Fisher test says the motif is **associated** with the inhibitor
  label in this dataset. Also not causation, and the dataset is one assay.
- The property check is a direction check on aggregates. A motif matching the
  literature is weak support; it does not validate any individual prediction.

Taken together these say the model is reading recognisable chemistry rather than
noise. That is the most this analysis supports.